<div style="text-align:center; font-size:50px; font-weight:700; margin-top:25px;">
Python pour Data Science - Projet - Groupe 4A
</div>

<div style="text-align:center; font-size:40px; font-weight:700; margin-top:25px;">
Prédiction des prix immobiliers
</div>

<br>

<div style="display:flex; justify-content:space-between; margin-top:35px; font-size:20px;">

  <div style="width:45%;">
    <div style="font-size:22px; font-weight:700; margin-bottom:10px;">Étudiants :</div>
    <div>• Yassine MELLOUL</div>
    <div>• Amira BARHOUMI</div>
    <div>• Antoine FOUCART</div>
  </div>

  <div style="width:45%; text-align:right;">
    <div style="font-size:22px; font-weight:700; margin-bottom:10px;">Chargé de TD :</div>
    <div>Julien PRAMIL</div>
  </div>

</div>

<br>
<hr>

# **1 - DESCRIPTION ET PROBLEMATIQUE**

Dans un contexte où les prix immobiliers sont fortement hétérogènes selon les territoires, le marché du logement en France présente de fortes disparités liées à la localisation, à 

l’attractivité des zones et aux dynamiques socio-économiques locales. Cette variabilité rend complexe la compréhension des mécanismes de formation des prix. Ainsi, la problématique 

centrale de ce projet est d’identifier et quantifier les facteurs influençant le prix de l’immobilier à l’échelle des communes. Nous nous concentrons en particulier sur des déterminants 

socio-économiques tels que la densité de population, le taux de chômage, le revenu moyen et le taux de pauvreté, afin d’évaluer leur impact sur les niveaux de prix. Pour cela, nous 

mobilisons deux sources de données principales : la base DVF (Demandes de Valeurs Foncières), qui recense les transactions immobilières en France , et les données socio-économiques de 

l’INSEE . Les jeux de données utilisés sont accessibles respectivement sur data.gouv.fr et insee.fr, via les liens suivants : 
https://www.data.gouv.fr/datasets/demandes-de-valeurs-foncieres et https://www.insee.fr/fr/statistiques/5359146 .

In [ ]:
# IMPORTATION
# modules
import pandas as pd
import numpy as np

# fonctions
from src.data.collect import load__data_url_zip_txt, load_insee_dossier_complet,load_logements_sociaux
from src.data.clean import filter_dvf_columns, compute_prix_m2, preprocess_insee, merge_all
from src.data.stats_desc import univariate_numeric_analysis, carte_dep_communes_cartiflette, analyse_bivariée_quant_quant
#URL
URL_DVF = "https://static.data.gouv.fr/resources/demandes-de-valeurs-foncieres/20260405-002321/valeursfoncieres-2025.txt.zip"
URL_INSEE = "https://www.insee.fr/fr/statistiques/fichier/5359146/dossier_complet.zip"
URL_LOGEMENTS_SOCIAUX = "https://www.data.gouv.fr/api/1/datasets/r/b0d30277-3a14-4673-a988-2fa6c11e030c"





# **2 - COLLECTE DE DONNEES ET NETTOYAGE**

## **2-1 COLLECTE**

In [ ]:
# collecte de la base dvf
dvf = load__data_url_zip_txt(URL_DVF)

# aperçu des données brutes
summary_dvf = dvf.dtypes.to_frame(name="type")
summary_dvf["nb_valeurs_manquantes"] = dvf.isna().sum()

summary_dvf = summary_dvf.reset_index().rename(columns={"index": "variable"})

n_rows, n_cols = dvf.shape

print(f"Nombre de lignes : {n_rows}")
print(f"Nombre de colonnes : {n_cols}")

print(summary_dvf)
print(dvf.head(2))

La base contient 3,7 millions de lignes et 43 colonnes. On observe que de nombreuses colonnes sont quasi entièrement vides (identifiants, lots, articles CGI). Les colonnes pertinentes pour notre étude sont le prix, la surface, le type de bien et la localisation. On conserve uniquement les ventes de maisons et d'appartements avec un prix et une surface strictement positifs, ce qui réduit la base à environ 1,1 million de transactions.

In [ ]:
# collecte de la base INSEE
insee = load_insee_dossier_complet(URL_INSEE)

# aperçu des données brutes
summary_insee = insee.dtypes.to_frame(name="type")
summary_insee["nb_valeurs_manquantes"] = insee.isna().sum()

summary_insee = summary_insee.reset_index().rename(columns={"index": "variable"})

n_rows, n_cols = insee.shape

print(f"Nombre de lignes : {n_rows}")
print(f"Nombre de colonnes : {n_cols}")

print(summary_insee)
print(insee.head(5))

La base contient 34 988 communes et 7 variables. Le taux de pauvreté (TP6021) présente 30 591 valeurs manquantes sur 34 988 (87%), en raison du secret statistique appliqué aux communes de moins de 1 000 ménages. Cette variable est donc exclue de la modélisation. Le revenu médian (MED21) est au format texte avec des virgules comme séparateur décimal, il nécessite une conversion.

## **2-1 NETTOYAGE**

### **Base DVF**
Le nettoyage des données DVF vise à construire un jeu de données cohérent et exploitable pour la modélisation du prix immobilier.

Dans un premier temps, seules les variables pertinentes sont conservées : prix de transaction, localisation (code postal, commune, département, code commune), caractéristiques du bien (type de local, surface bâtie) et type de mutation. Les variables inutiles ou trop détaillées sont supprimées.

Ensuite, plusieurs transformations sont appliquées :
- conversion du prix en format numérique (suppression des espaces, gestion des virgules),
- harmonisation des types (variables qualitatives en `string`),
- standardisation des codes géographiques (zfill pour obtenir des codes à 2 et 5 chiffres).

Un filtrage est réalisé pour ne conserver que les observations pertinentes :
- uniquement les ventes,
- prix et surface strictement positifs,
- biens de type *Maison* ou *Appartement*.

Les valeurs manquantes sont supprimées afin d’assurer la qualité des données.

Enfin, une variable dérivée **prix au m²** est calculée comme le ratio entre la valeur foncière et la surface bâtie.

Ce processus permet d’obtenir un jeu de données propre, homogène et directement utilisable pour l’analyse et la modélisation.



In [ ]:
# Nettoyage
dvf = filter_dvf_columns(dvf)
# Ajout de la varible prix du metre carré
dvf = compute_prix_m2(dvf)
# aperçu des données tratées
summary_dvf_f = dvf.dtypes.to_frame(name="type")
summary_dvf_f["nb_valeurs_manquantes"] = dvf.isna().sum()

summary_dvf_f = summary_dvf_f.reset_index().rename(columns={"index": "variable"})

n_rows, n_cols = dvf.shape

print(f"Nombre de lignes : {n_rows}")
print(f"Nombre de colonnes : {n_cols}")

print(summary_dvf_f)
print(dvf.head(5))

### **Base INSEE**
La fonction `preprocess_insee` a pour objectif de transformer un DataFrame INSEE brut en un jeu de données propre, standardisé et exploitable pour l’analyse statistique ou la modélisation.

Dans un premier temps, une copie du DataFrame est créée afin d’éviter toute modification directe des données d’origine. Cette étape garantit une approche non destructive du traitement.

Ensuite, les colonnes sont renommées pour améliorer leur lisibilité et leur cohérence. Les noms techniques issus de l’INSEE (comme `CODGEO`, `MED21` ou `TP6021`) sont remplacés par des appellations explicites telles que `code_commune`, `mediane_niveau_vie` ou `taux_pauvrete`. Cette standardisation facilite l’écriture du code et la compréhension des variables.

La fonction procède ensuite à la conversion des types de données. Le code commune et la médiane du niveau de vie sont convertis en type `string`, car ils correspondent à des identifiants ou des valeurs textuelles. Le taux de pauvreté subit un nettoyage plus avancé : les valeurs non exploitables comme `"s"` ou `"nd"` sont remplacées par des valeurs manquantes (`NaN`), les virgules sont converties en points pour respecter le format numérique, puis la variable est transformée en nombre et divisée par 100 afin d’obtenir un taux compris entre 0 et 1.

À partir des variables existantes, de nouveaux indicateurs sont créés. Le taux de chômage est calculé en rapportant le nombre de chômeurs de 15 à 64 ans à la population totale, ce qui fournit une mesure simple de la pression du chômage. La densité de population est également construite en divisant la population par la superficie, permettant d’évaluer le degré de concentration des habitants sur le territoire.

Enfin, toutes les lignes contenant des valeurs manquantes sont supprimées avec `dropna()`, afin d’obtenir un jeu de données entièrement complet et cohérent pour les analyses futures.

In [ ]:
# Nettoyage
insee = preprocess_insee(insee)

# aperçu des données traitées
summary_insee_f = insee.dtypes.to_frame(name="type")
summary_insee_f["nb_valeurs_manquantes"] = insee.isna().sum()

summary_insee_f = summary_insee_f.reset_index().rename(columns={"index": "variable"})

n_rows, n_cols = insee.shape

print(f"Nombre de lignes : {n_rows}")
print(f"Nombre de colonnes : {n_cols}")

print(summary_insee_f)
print(insee.head(5))

### **Fusion des jeux de données selon code commune**

In [ ]:
# merge
log_soc=load_logements_sociaux(URL_LOGEMENTS_SOCIAUX)
data_final = merge_all(dvf, insee,log_soc)
# aperçu
print(data_final.head(6))
print(data_final.shape)

# **3 - ANALYSE DESCRIPTIVE**

## **3-1 - Univarié**

### **Valeur foncière**

In [ ]:
stats = univariate_numeric_analysis(data_final,"valeur_fonciere" )

### **prix du mètre carré**

In [ ]:
stats2 = univariate_numeric_analysis(data_final,"prix_m2" )

In [ ]:
q05 = data_final["prix_m2"].quantile(0.05)
q95 = data_final["prix_m2"].quantile(0.95)
data_final = data_final[data_final["prix_m2"] > q05]
data_final = data_final[data_final["prix_m2"] < q95]

statsq = univariate_numeric_analysis(data_final,"prix_m2" )

### **densité de population**

In [ ]:
stats3 = univariate_numeric_analysis(data_final,"densite" )

### **Superficie de communes**

In [ ]:
stats4 = univariate_numeric_analysis(data_final,"superficie" )

## **3-2 - Bivarié**

In [ ]:
analyse_bivariée_quant_quant(data_final,["prix_m2", "densite", "taux_chomage", "mediane_niveau_vie"])

In [ ]:
carte_dep_communes_cartiflette(
    df=data_final,
    col="taux_chomage",
    code_dep=75
)

# **4 - MODELISATION**

L'objectif de cette partie est de prédire le prix au m² des biens immobiliers à partir des caractéristiques du bien (DVF) et du contexte socio-économique de la commune (INSEE). Nous comparons deux approches : une régression linéaire comme modèle de référence, et un Random Forest comme modèle principal.

### Imports :

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score
 
from src.model.train import prepare_features, split_data, train_linear_regression, train_random_forest, train_gradient_boosting
from src.model.evaluate import evaluate_model, plot_feature_importance, plot_predictions, plot_residuals

### Chargement des données :

### Préparation et split :

On sépare les données en 80% pour l'entraînement et 20% pour le test. Le modèle apprend sur le jeu d'entraînement et on évalue ses performances sur le jeu de test, c'est-à-dire des données qu'il n'a jamais vues. Cela permet de mesurer sa capacité à généraliser.

In [ ]:
X, y = prepare_features(data_final)
X_train, X_test, y_train, y_test = split_data(X, y)

print(f"Train : {X_train.shape[0]} lignes")
print(f"Test  : {X_test.shape[0]} lignes")
print(X_train.isnull().sum())
X.dropna()

### Régression linéaire :

La régression linéaire suppose une relation proportionnelle entre chaque variable et le prix au m² : prix = a × surface + b × revenu + ... C'est un modèle simple et interprétable grâce à ses coefficients. Il sert de baseline : si un modèle plus complexe ne fait pas mieux, c'est qu'il n'apporte rien.

In [ ]:
reg = train_linear_regression(X_train, y_train)
y_pred_reg = reg.predict(X_test)

res_reg = evaluate_model(y_test, y_pred_reg, "Régression Linéaire")
 

# Coefficients
coefs = pd.DataFrame({
    "variable": X.columns,
    "coefficient": reg.coef_
}).sort_values("coefficient", ascending=False)
coefs

### Random Forest :

Le Random Forest est un ensemble de 100 arbres de décision qui votent ensemble pour la prédiction finale. Contrairement à la régression linéaire, il capture les relations non-linéaires et les interactions entre variables (par exemple : grande surface dans une commune riche ≠ grande surface dans une commune pauvre). Il ne nécessite pas de normalisation des données.

In [ ]:
rf = train_random_forest(X_train, y_train)
y_pred_rf = rf.predict(X_test)
 
res_rf = evaluate_model(y_test, y_pred_rf, "Random Forest")

In [ ]:
gb = train_gradient_boosting(X_train, y_train)
y_pred_gb = gb.predict(X_test)
 
res_gb = evaluate_model(y_test, y_pred_gb, "Gradient Boosting")
 

### Comparaison des modèles :
On compare les deux modèles sur trois métriques :

MAE : erreur moyenne en valeur absolue, intuitive (en €/m²)
RMSE : pénalise davantage les grosses erreurs, pertinent pour l'immobilier où se tromper de 50 000€ est grave
R² : proportion de la variance des prix expliquée par le modèle (1 = parfait, 0 = inutile)


In [ ]:
resultats = pd.DataFrame([res_reg, res_rf, res_gb])
resultats

### Prédictions vs réalité :
Ce graphique compare les prix prédits aux prix réels. Plus les points sont proches de la diagonale rouge (prédiction = réalité), meilleur est le modèle. On observe que le Random Forest concentre davantage ses prédictions autour de la diagonale, confirmant sa supériorité.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(y_test, y_pred_reg, alpha=0.3, s=10)
axes[0].plot([0, 15000], [0, 15000], color="red", linestyle="--")
axes[0].set_title(f"Régression Linéaire (R² = {r2_reg:.3f})")
axes[0].set_xlabel("Prix réel (€/m²)")
axes[0].set_ylabel("Prix prédit (€/m²)")

axes[1].scatter(y_test, y_pred_rf, alpha=0.3, s=10)
axes[1].plot([0, 15000], [0, 15000], color="red", linestyle="--")
axes[1].set_title(f"Random Forest (R² = {r2_rf:.3f})")
axes[1].set_xlabel("Prix réel (€/m²)")
axes[1].set_ylabel("Prix prédit (€/m²)")

plt.tight_layout()
plt.show()